# 01 — Label Audit

Audits the SDSS DR19 merged catalogue (`DATA7901_DR19_merged.csv`) before any ML work.
Covers label construction, vote fraction distributions, sentinel value counts, feature
distributions, and working-set availability (images + spectra).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

%matplotlib inline

# Add project root to sys.path so src/ is importable
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR   = PROJECT_ROOT / 'input' / 'tables'
IMG_DIR    = PROJECT_ROOT / 'input' / 'images'
SPEC_DIR   = PROJECT_ROOT / 'input' / 'spectra'
CSV_PATH   = DATA_DIR / 'DATA7901_DR19_merged.csv'

print('Project root:', PROJECT_ROOT)
print('CSV path exists:', CSV_PATH.exists())

In [ ]:
from src.data.rename import apply_renames
from src.data.labels import make_labels
from src.data.clean import (
    TABULAR_FEATURES, engineer_features, handle_missing, get_feature_cols
)

df_raw = pd.read_csv(CSV_PATH, low_memory=False)
df = apply_renames(df_raw)

print(f'Shape after rename: {df.shape}')
print('\nFirst 3 columns of interest:')
cols_of_interest = ['objid', 'spec_z', 'spec_mjd', 'p_el', 'p_cs', 'p_mg', 'nvote_tot', 'p_dk']
present = [c for c in cols_of_interest if c in df.columns]
print(df[present[:3]].head(3))

from src.data.rename import apply_renames
from src.data.labels import make_labels
from src.data.features import (
    TABULAR_FEATURES, engineer_features, handle_missing, get_feature_cols
)

In [ ]:
labeled = make_labels(df)

frac_labeled = len(labeled) / len(df)
print(f'\nLabeled fraction of total rows: {frac_labeled:.3f} ({len(labeled):,} / {len(df):,})')

## 2. Vote Fraction Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

vote_cols = [
    ('p_el', 'Elliptical fraction', 'steelblue'),
    ('p_cs', 'Combined spiral fraction', 'darkorange'),
    ('p_mg', 'Merger fraction', 'seagreen'),
]

for ax, (col, title, color) in zip(axes, vote_cols):
    ax.hist(df[col].dropna(), bins=50, color=color, edgecolor='white', linewidth=0.4)
    ax.axvline(0.60, color='red', linestyle='--', linewidth=1.2, label='threshold=0.60')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Vote fraction')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

fig.suptitle(
    'Galaxy Zoo vote fraction distributions\n'
    '(bimodal peaked at 0 and 1 = clean labels)',
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df['nvote_tot'].dropna(), bins=80, color='slategray', edgecolor='white', linewidth=0.4)

for threshold, color in [(10, 'gold'), (20, 'red'), (30, 'darkred')]:
    ax.axvline(threshold, color=color, linestyle='--', linewidth=1.4, label=f'threshold={threshold}')

ax.set_xlabel('nvote_tot')
ax.set_ylabel('Count')
ax.set_title('Total Galaxy Zoo vote count distribution')
ax.set_xlim(0, df['nvote_tot'].quantile(0.999))
ax.legend()
plt.tight_layout()
plt.show()

print('Labeled galaxies surviving each nvote_tot threshold (after other filters):')
base = df[(df['class'] == 'GALAXY') & (df['sdssPrimary'] == 1) & (df['p_dk'] < 0.30)]
for t in [10, 20, 30]:
    n = len(base[base['nvote_tot'] >= t])
    print(f'  nvote_tot >= {t}: {n:,} rows')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df['p_dk'].dropna(), bins=60, color='mediumpurple', edgecolor='white', linewidth=0.4)
ax.axvline(0.30, color='red', linestyle='--', linewidth=1.4, label='max_dk=0.30')
ax.set_xlabel('p_dk (don\'t know fraction)')
ax.set_ylabel('Count')
ax.set_title('Distribution of p_dk — artifact/ambiguous image indicator')
ax.legend()
plt.tight_layout()
plt.show()

pct_excluded = 100 * (df['p_dk'] >= 0.30).sum() / len(df)
print(f'Rows excluded by p_dk >= 0.30: {(df["p_dk"] >= 0.30).sum():,} ({pct_excluded:.1f}% of full catalogue)')

## 3. Sentinel Value Audit

In [ ]:
sentinel_counts = {}
for col in TABULAR_FEATURES:
    if col in df.columns and df[col].dtype != object:
        n_sentinel = (df[col] < -100).sum()
        if n_sentinel > 0:
            sentinel_counts[col] = n_sentinel

if sentinel_counts:
    sentinel_df = (
        pd.Series(sentinel_counts, name='sentinel_count')
        .sort_values(ascending=False)
        .reset_index()
        .rename(columns={'index': 'column'})
    )
    sentinel_df['pct_of_rows'] = (sentinel_df['sentinel_count'] / len(df) * 100).round(2)
    print(f'Columns with sentinel values (< -100) — {len(sentinel_df)} of {len(TABULAR_FEATURES)} feature cols:\n')
    print(sentinel_df.to_string(index=False))
else:
    print('No sentinel values found in TABULAR_FEATURES columns.')

## 4. Feature Distributions

In [ ]:
labeled_clean = handle_missing(labeled)
labeled_clean = engineer_features(labeled_clean)

print(f'Rows after handle_missing + engineer_features: {len(labeled_clean):,}')

# Sample 3000 points for speed
rng = np.random.default_rng(42)
sample = labeled_clean.sample(n=min(3000, len(labeled_clean)), random_state=42)

color_map = {'elliptical': 'red', 'spiral': 'steelblue', 'merger': 'green'}

fig, ax = plt.subplots(figsize=(8, 6))
for lbl, grp in sample.groupby('label'):
    ax.scatter(
        grp['color_g_r'],
        grp['concentration'],
        c=color_map.get(lbl, 'gray'),
        label=lbl,
        alpha=0.45,
        s=12,
        edgecolors='none'
    )

ax.set_xlabel('g - r colour (dered_g - dered_r)')
ax.set_ylabel('Concentration (petroR90_r / petroR50_r)')
ax.set_title('Colour vs Concentration by morphological class (3k sample)')
ax.legend(title='label', markerscale=2)
ax.set_ylim(0, 8)
plt.tight_layout()
plt.show()

## 5. Working Set Summary

In [ ]:
def check_working_set(labeled_df, img_dir, spec_dir):
    df = labeled_df.copy()

    img_dir  = Path(img_dir)
    spec_dir = Path(spec_dir)

    df['has_image'] = df['objid'].apply(
        lambda x: (img_dir / f'{x}.jpeg').exists()
    )

    df['spec_filename'] = df.apply(
        lambda r: f"spec-{int(r['plate'])}-{int(r['spec_mjd'])}-{int(r['fiberid']):04d}.fits",
        axis=1
    )
    df['has_spectrum'] = df['spec_filename'].apply(
        lambda fn: (spec_dir / fn).exists()
    )

    df['has_both'] = df['has_image'] & df['has_spectrum']
    return df


ws = check_working_set(labeled, IMG_DIR, SPEC_DIR)

total      = len(ws)
n_image    = ws['has_image'].sum()
n_spectrum = ws['has_spectrum'].sum()
n_both     = ws['has_both'].sum()

print('Working set availability:')
print(f'  Total labeled galaxies : {total:,}')
print(f'  has_image              : {n_image:,} ({100*n_image/total:.1f}%)')
print(f'  has_spectrum           : {n_spectrum:,} ({100*n_spectrum/total:.1f}%)')
print(f'  has_both               : {n_both:,} ({100*n_both/total:.1f}%)')
if n_image == 0 and n_spectrum == 0:
    print('\nNote: images and spectra have not been downloaded yet — run')
    print('  src/data/download_images.py and src/data/download_spectra.py')

In [ ]:
keep_cols = ['objid', 'ra', 'dec', 'plate', 'spec_mjd', 'fiberid',
             'label', 'label_id', 'p_el', 'p_cs', 'p_mg', 'nvote_tot']
present_cols = [c for c in keep_cols if c in labeled.columns]

labeled_index = labeled[present_cols].copy()

out_path = DATA_DIR / 'labeled_index.csv'
labeled_index.to_csv(out_path, index=False)

print(f'Saved labeled_index.csv: {len(labeled_index):,} rows x {len(present_cols)} columns')
print(f'Path: {out_path}')
print('\nColumn dtypes:')
print(labeled_index.dtypes)